# 1.Introdução

## Engenharia de Variaveis Pipeline

Neste notebook, será estudado aplicado técnicas de engenharia de featura que foram selecionadas após a etapa de EDA, aqui será criado todo o pre-processamento e aplicação dentro de um pipeline, buscando otimizar a execução e a legibilidade do modelo, também será muito util para o deploy, posteriormente,

# 2. Escopo e Estratégia de Engenharia de Variáveis

Nesta etapa inicial da engenharia de variáveis, o foca será a base 'application_train.csv', considerando um escopo incremental para o projeto.

As principais diretrizes adotadas incluem:
- Manutenção de variáveis com potencial informativo
- Tratamento consicente de valores ausentes
- Preparação das variáveis para modelos lineares
- Priorização de interpretabilidade

# 3. Importação das bibliotecas e configurações iniciais

In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 4. Carregamento dos dados 
Os dados são carregados a partir  da pasta 'data/' que não é versionada no repositorio seguindo boas práticas da engenharia de dados.

In [18]:
# CARREGABDI A BASE DE DADOS JÁ ANALISADA
application_train = pd.read_csv('../data/application_train.csv', encoding = 'iso-8859-1')
application_train.shape

(307511, 122)

# 5. Separação as Features e o Target
Nesta seção, a variável target ('TARGET') é separada das variáveis explicativas, garantido clareza no pipeline de preparação dos dados e evitando vazamento de informação.

In [63]:
# SEPARANDO AS FEATURES E O TARGET
X = application_train.drop(columns = ['TARGET'])
Y = application_train['TARGET']

# EXIBINDO O FORMATO DAS FEATURE E DO TARGET
print('features: {},\ntarget: {}'.format(X.shape, Y.shape))

# COPIANDO O RESULTADO PARA MODIFICAÇÕES POSTERIORES
features = X.copy()
target = Y.copy()

features: (307511, 121),
target: (307511,)


# 6. Separação das Variaveis

Nesta etapa, pós fazer a separação das features e do target. Estarei separando as varaiveis por tipos (float, int, categoricas)

In [64]:
# EXIBINDO QUAIS OS TIPOS DAS FEATURES
features.dtypes.unique()

# SEPARANDO AS FEATURES ENTRE NUMERICAS E CATEGORICAS
numericas = features.select_dtypes(['int64','float64']).columns
categoricas = features.select_dtypes(['O']).columns

# EXIBINDO O FORMATO DE CADA TIPO CRIADO
print('numericas: {}\ncategoricas: {}'.format(numericas.shape, categoricas.shape))

numericas: (105,)
categoricas: (16,)


# 7. Criando o pipeline
Nesta etapa, foi criado o pipeline responsável por aplicar todo o pre-processamento às features. Idealizado através da investigação feita na EDA, foram escolhidas algumas técnicas e implementadas por meio de duas classes e aplicação do ColumnTransformer.
- MissingFlagTransformer: Responsável por realizar a criação das variáveis Flags
- FeatureEngineeringTransformer: Responsável pela etapa de Feature Creation.
- ColumnTransformer: Imputação da mediana nas features númericas, imputação da nova categoria 'unknown' e criação de variaveis dummys para as features categoricas.

## 7.1 MissingFlagTransformer
Essa classe foi implementada com inteção de criar as varáveis flags onde as colunas que possuem valores "NaN". Essa estrategia foi aplicada porque as muitas vezes os registros como NaN, podem significar falta de aplicabilidade aquela caracteristica do cliente, essa informação pode ser útil para ajudar o modelo a classificar a probabilidade do cliente ser ou não ser inadimplente.

In [65]:
# CRIA A CLASSE QUE SERÁ RESPONSÁVEL PELA CRIAÇÃO DAS VARIAVEIS FLAGS ONDE HOUVER VALOR NAN
class MissingFlagTransformer(BaseEstimator, TransformerMixin):
    
    # CRIA A FUNÇÃO FIT, RESPONSÁVEL POR RECEBER O DATASET, COLETAR O NOME DAS COLUNAS E ARMAZENAR NA MEMORIA
    def fit(self, features, target = None):
        
        # ARMAZENA NA MEMORIA (SELF) A LISTA DAS COLUNAS COM VALORES NULOS
        self.colunas_com_missing_ = (features.columns[features.isnull().mean() >0])
        
        # RETORNA ESSA LISTA NA MEMORIA
        return self
                 # CRIA A FUNÇÃO QUE SERÁ RESPONSAVEL POR REALIZAR A CRIAÇÃO DAS FLAGS A PARTIR DA LISTA DENTRO DE SELF
    def transform(self, features):
        
        # CRIA UMA COPIA DO DATASET
        features = features.copy()

        # CRIA A ESTRUTURA REPETIÇÃO QUE IRÁ ITERAR DENTRO DA MEMORIA E BUSCAR UMA COLUNA POR VEZ
        for coluna in self.colunas_com_missing_:
                
            # SE ESTIVER, CRIAR AS COLUNAS FLAGS COM BASE NO NOME DA COLUNA ANTIGO E SE OS VALORES FOREM NAN APLICAR 1 = SIM, 0 = NÃO
            # ATRAVÉS DO ASTYPE(INT)
            features[f'{coluna}_missing_flag'] = features[coluna].isnull().astype(int)
                
                # RETORNA O DATAFRAME COM FLAGS
        return features

        

## 7.2  Feature EngineeringTransformer
Com base na distribuição das variáveis que foram analisadas durante a EDA, cheguei a conclusão que por conta da diversidade financeira de clientes, existem muitos clientes que são outliers e existe uma longa cauda a direita (skew). A decisão de criar novas colunas aplicando a escala logaritma, foi apoiada pela função skew(), as variaveis que foran analisadas possuiam um coeficiente > 1,a ideia é tentar melhorar a estabilidade do modelo evitando enviesamento.

Logo após essa decisão, implementei novas variaveis utilizando a razão entre as features analisadas e features já existentes. Gerando assim:
- RATIO_CREDIT_INCOME = Razão de Crédito-Renda, Representa o tamanho da dívida/credito solicitado em relação ao ganho dototal do cliente, indica quantas vezes a redna anual dele o crédito solicitado representa.
- RATIO_ANNUITY_INCOME = Capacidade de Pagamento, mede a fatia da renda mensal/anual que será gasta puramente para pagar a parcela (Annuity) do emprestimo
- CREDIT_PER_PERSON = Cérdito per Capita, Avalia o tamano do crédito dividindo pelo tamanho da familia. Um cre´dito alto para uma família muito grande pode significar um risco doluído ou concentrado a depender a estrutura familiar.
- INCOME_PER_PERSON = Renda Per Capita Familiar = É o indicador real de bem-estar financeiro.
- RATIO_CREDIT_GOODS = Razão Empréstimo-Valor (Garantia), Indica a porcentagem do valor do bem que está sendo financiada. Se o LTV for maior que 1, significa que o cliente está pegando um empréstimo maior do que o preço do próprio bem que ele quer comprar.

Foi substituido também a quantidade de familiares de todos os clientes do banco, de 0 para 1, para evitar erro na divisão, pois ela foi a coluna responsável por gerar uma das features 'CREDIT_PER_PERSON'.

Dentro da classe, também existe um trecho de código utilizado para automatizar a remoção dos nomes das features que serão geradas durante a aplicação do Pipeline Final =  ' features.columns = [col.split('__')[-1] for col in features.columns]'

In [75]:
# CLASSE RESPONSAVEL PELA ETAPA DE FEATURE CREATION
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    
    # RECEBE O DATASET E ARMAZENA NA MEMORIA
    def fit(self, features, target=None):
        return self
        
    # CRIA AS NOVAS COLUNAS
    def transform(self, features):
        features = features.copy()
        
       # REMOVE OS PREXIFOS NOS NOMES DAS COLUNAS QUE SÃO CRIADOS A CADA PASSO APLICADO DO PIPELINE PELO COLUMNTRANSFORMER
        features.columns = [
            col.split('__')[-1] for col in features.columns
        ]

        # LOG TRANSFORM (Corrigido de 'X' para 'features')
        features['AMT_INCOME_TOTAL_log'] = np.log1p(features['AMT_INCOME_TOTAL'])
        features['AMT_CREDIT_log'] = np.log1p(features['AMT_CREDIT'])
        features['AMT_GOODS_PRICE_log'] = np.log1p(features['AMT_GOODS_PRICE'])
        features['AMT_ANNUITY_log'] = np.log1p(features['AMT_ANNUITY'])

        # EVITANDO DIVISÃO POR ZERO
        features['CNT_FAM_MEMBERS'] = features['CNT_FAM_MEMBERS'].replace(0, 1)

        # FEATURE CREATION
        features['RATIO_CREDIT_INCOME'] = features['AMT_CREDIT'] / (features['AMT_INCOME_TOTAL'] + 1)
        features['RATIO_ANNUITY_INCOME'] = features['AMT_ANNUITY'] / (features['AMT_INCOME_TOTAL'] + 1)
        features['CREDIT_PER_PERSON'] = features['AMT_CREDIT'] / (features['CNT_FAM_MEMBERS'] + 1)
        features['INCOME_PER_PERSON'] = features['AMT_INCOME_TOTAL'] / (features['CNT_FAM_MEMBERS'] + 1)
        features['RATIO_CREDIT_GOODS'] = features['AMT_CREDIT'] / (features['AMT_GOODS_PRICE'] + 1)
        
        # RETORNA O DATASET
        return features

## 7.3 CRIAÇÃO DOS PIPELINES
Nesta seção foi implementado a criação dos pipelines que serão utilizados para compor o pipeline final. 
- pipeline_numerico: Respomsável por aplicar a mediana em todas as features númericas que possuem valores NaN
- pipeline_categorico: Responsável por aplicar a nova categoria 'Unknown' onde os valores forem NaN e também faz a criação das variaveis dummys.
- preprocessor: Responsável por aninha os dois pipelies citados (pipeline_numerico e pipeline_categorico), utiliza o parametro remainder = 'passthrough' para evitiar que o pipeline final descarte as colunas que serão criadas durante a aplicação dos outros passos, por fim aplica a função .set_output onde força ao ColumnTransfomer a devolver a base de dados em um formato de dataset.

In [76]:
# CRIA O PIPELINE PARA AS FEATURES NUMERICAS IMPUTANDO A MEDIANA ONDE FOR NAN
pipeline_numerico = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median'))
])

# CRIA O PIPELINE CATEGORICO IMPUTANDO A NOVA CATEGORIA UNKNOW ONDE FOR NAN
pipeline_categorico = Pipeline([
    # IMPUTA UMA NOVA CATEGORIA PARA AS FEATURES CATEGORICAS
    ('imputer', SimpleImputer(strategy = 'constant', fill_value = 'Unknown')),
    
    # UTILIZA O ONEHOTENCODER PARA CRIAR AS VARIAVEIS DUMMYS DROPANDO A 1 COLUNA E
    # UTILIZANDO O SPARSE_OUTPUT = FALSE, PARA NÃO GERAR UMA MATRIZ SPARSA COMPRIMIDA
    ('encoder', OneHotEncoder(drop = 'first', handle_unknown = 'ignore', sparse_output = False))
]) 

# UNINDO NO COLUMNTRANSFORMER E GARANTINDO QUE QUEM NÃO FOR LISTADO PASSE ADIANTE
preprocessor = ColumnTransformer(
    transformers = [
        ('numericas', pipeline_numerico, numericas),
        ('categoricas', pipeline_categorico, categoricas)
    ],
    remainder = 'passthrough' # ESSE PARAMETRO, FAZ COM QUE AS COLUNAS QUE RECEBEREM O PRE-PROCESSAMENTO, NÃO SEJAM EXCLUIDAS
)
# FORÇA O COLUMNTRANSFORMER A DEVOLVER A BASE DE DADOS EM UM DATASET PANDAS
preprocessor.set_output(transform = 'pandas')

,transformers,"[('numericas', ...), ('categoricas', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 7.4 Aplicação do Pipeline Final
Aqui foi feito a aplicação do pipeline final, utilizando a função Pipeline, os passos foram compostos pelas classes anteriormente criadas (MissingFlagTransformer e FeatureEngineeringTransformer) e o preprocesser, que é onde foi feito a "concatenação" dos pipelines (pipeline_numerico e pipeline_categorico).

In [77]:
# CRIA O PIPELINE E PASSA A ORDEM DO PRE-PROCESSAMENTO,
# 1. CRIA AS FLAGS FEATURES
# 2. APLICA O PRE-PROCESSAMENTO DOS PIPELINE NUMERICOS E CATEGORICOS
# 3. CRIA AS NOVAS COLUNAS APLICANDO A ESCALA LOGARITMA E CRIA AS NOVAS COLUNAS BASEADA NA RAZÃO DAS COLUNAS ANALSIADAS
pipeline_preprocessamento = Pipeline(steps =[
    # APLICA AS FUNÇÕES DA CLASSE MISSING FLAGS
    ('missing_flags', MissingFlagTransformer()),
    # APLICA O PRE-PROCESSAMENTO DEFINIDO DENTRO DOS PIPELINE ANINHADOS
    ('preprocessor', preprocessor),
    # APLICA AS FUNÇÕES DA CLASSE FEATURE ENGINEERING
    ('feature_engineering', FeatureEngineeringTransformer())
])

## 7.5 Base Transformada
Finalizando a seção, foi aplicado o pipeline final, utilizando a função fit_transform() nas features que foram analisadas e estãos sendo trabalhadas.

In [78]:
# APLICA O PRE-PROCESSAMENTO AS FEATURES
base_transformada = pipeline_preprocessamento.fit_transform(features)

# EXIBINDO
display(base_transformada)

,SK_ID_CURR,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,AMT_REQ_CREDIT_BUREAU_YEAR_missing_flag,AMT_INCOME_TOTAL_log,AMT_CREDIT_log,AMT_GOODS_PRICE_log,AMT_ANNUITY_log,RATIO_CREDIT_INCOME,RATIO_ANNUITY_INCOME,CREDIT_PER_PERSON,INCOME_PER_PERSON,RATIO_CREDIT_GOODS
0,100002.0,0.0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461.0,-637.0,-3648.0,...,0,12.218500,12.915581,12.768544,10.114619,2.007879,0.121977,203298.75,101250.0,1.158394
1,100003.0,0.0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765.0,-1188.0,-1186.0,...,0,12.506181,14.072865,13.937287,10.482892,4.790732,0.132216,431167.50,90000.0,1.145198
2,100004.0,0.0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046.0,-225.0,-4260.0,...,0,11.119898,11.813037,11.813037,8.817446,1.999970,0.099999,67500.00,33750.0,0.999993
3,100006.0,0.0,135000.0,312682.5,29686.5,297000.0,0.008019,-19005.0,-3039.0,-9833.0,...,1,11.813037,12.652947,12.601491,10.298481,2.316150,0.219898,104227.50,45000.0,1.052799
4,100007.0,0.0,121500.0,513000.0,21865.5,513000.0,0.028663,-19932.0,-3038.0,-4311.0,...,0,11.707678,13.148033,13.148033,9.992711,4.222187,0.179961,256500.00,60750.0,0.999998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251.0,0.0,157500.0,254700.0,27558.0,225000.0,0.032561,-9327.0,-236.0,-8456.0,...,1,11.967187,12.447846,12.323860,10.224084,1.617133,0.174970,127350.00,78750.0,1.131995
307507,456252.0,0.0,72000.0,269550.0,12001.5,225000.0,0.025164,-20775.0,365243.0,-4388.0,...,1,11.184435,12.504513,12.323860,9.392870,3.743698,0.166685,134775.00,36000.0,1.197995
307508,456253.0,0.0,153000.0,677664.0,29979.0,585000.0,0.005002,-14966.0,-7921.0,-6737.0,...,0,11.938200,13.426408,13.279369,10.308286,4.429148,0.195940,338832.00,76500.0,1.158398
307509,456254.0,0.0,171000.0,370107.0,20205.0,319500.0,0.005313,-11961.0,-4786.0,-2562.0,...,0,12.049425,12.821550,12.674516,9.913735,2.164356,0.118157,123369.00,57000.0,1.158391
